# MLOps demo with climatology data

We'll be using the [NOAA GHCN-D](https://registry.opendata.aws/noaa-ghcn/) public dataset.

## Processing

This is what each instance in the cluster is doing - code we pass in applies to each instance in cluster

**Reducing the raw archive**

- Read only 4 of the raw columns — ID, DATE, ELEMENT, DATA_VALUE
- Keep only TMAX, TMIN, PRCP rows, dropping the other ~50 element types
- Keep only observations from 2015 onward (--start_year)
- Pivot long → wide: one row per station-day, one column per element
- Convert units out of the archive's tenths — tenths °C → °C, tenths mm → mm

**Station metadata join**

- Parse ghcnd-stations.txt by fixed-width offsets into ID, LATITUDE, LONGITUDE, ELEVATION
- Replace the -999.9 unknown-elevation sentinel with NaN
- Drop stations missing latitude/longitude
- Merge lat/lon/elevation onto each station-day row

**Feature engineering** (sorted by station then date; all lags/rollings grouped by station so nothing leaks across stations)

- tmax as the target
- TMAX lags at 1, 2, 3, and 7 days
- tmin_lag_1, prcp_lag_1
- 7- and 30-day rolling means of TMAX, computed on the already-shifted series so today never enters its own window
- tmax_delta_1 = tmax_lag_1 − tmax_lag_2
- tmax_anomaly_7 = tmax_lag_1 − tmax_roll_7
- doy_sin / doy_cos from day-of-year, so December and January are adjacent rather than 364 apart
- Drop all rows with NaNs, which removes each station's lag/rolling warm-up period

**Splitting and output**

- Chronological split by year: train < 2023, validation 2023, test ≥ 2024
- Drop the year helper column from each split
- Write headed CSVs with the target in column 0, one file per split per instance

That leaves 15 features. The split is chronological rather than random specifically because the lag and rolling features would otherwise leak future weather backwards into training rows.

In [ ]:
!uv run python ../../src/process.py --env process.ghcn.yaml

## Training

In [ ]:
!uv run python ../../src/train.py --env train.random_forest.yaml

In [ ]:
!uv run python ../../src/train.py --env train.xgboost.yaml

## Evaluate which trained model is best

In [ ]:
!uv run python ../../src/process.py --env process.evaluate.yaml

## Register best model

In [ ]:
!uv run python ../../src/register.py --env register.climatology.yaml

## Batch predictions

In [ ]:
!uv run python ../../src/deploy.py --env deploy.climatology.yaml

## Generate report

In [ ]:
!uv run python ../../src/process.py --env process.report.yaml

## Orchestrate all the steps above into a workflow

In [ ]:
!uv run python ../../src/orchestrate.py --env orchestrate.climatology.yaml